# GridIQ — NoSQL: MongoDB (document store)  *(Monica)*

Loads the staging series into MongoDB as **one document per hour** with nested measures, a fuel sub-document, and quality flags — then validates with a count, a sample document, indexes, and aggregation-pipeline queries. Mirrors the Neo4J validation so the two NoSQL stores are documented the same way.

- Tries a **real `mongod` 7.0** server (downloaded + run on localhost); falls back to **`mongomock`** (in-memory, pymongo-compatible) if the server can't start — so it always completes. The document, insert, index, and aggregation code is identical either way.
- Loads the **full 5-year series** (43,824 docs), matching the Neo4J graph.
- Writes `BI/mongodb_validation.txt` — paste it into a “mongodb output” doc next to “neo4j output.”

> To point at a real cluster instead (e.g. MongoDB Atlas), set `client = MongoClient("<your-uri>")` in the setup cell.

### Setup — connect to MongoDB (real server, else mongomock)

In [ ]:
!pip install -q pymongo mongomock pyarrow
import os, sys, time, glob, json, subprocess, urllib.request, tarfile
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
import numpy as np, pandas as pd

DRIVE_ROOT = Path('/content/drive/MyDrive/GridIQ_Final_Project_MS_ADS_Spring_2026')
STAGING_PARQUET = DRIVE_ROOT / 'data' / 'staging' / 'staging_ercot_hourly.parquet'
BI_DIR = DRIVE_ROOT / 'BI'; BI_DIR.mkdir(parents=True, exist_ok=True)

DB_NAME, COLL_NAME = 'gridiq', 'hourly_readings'

def start_real_mongo():
    binp = '/content/mongo/bin/mongod'
    if not os.path.exists(binp):
        try: subprocess.run('apt-get -qq install -y libcurl4 openssl', shell=True, timeout=150)
        except Exception: pass
        url = 'https://fastdl.mongodb.org/linux/mongodb-linux-x86_64-ubuntu2204-7.0.14.tgz'
        urllib.request.urlretrieve(url, '/content/mongo.tgz')
        os.makedirs('/content/mongo_x', exist_ok=True)
        with tarfile.open('/content/mongo.tgz') as t: t.extractall('/content/mongo_x')
        os.rename(glob.glob('/content/mongo_x/mongodb-*/')[0], '/content/mongo')
    os.makedirs('/content/mdb', exist_ok=True)
    subprocess.Popen([binp, '--dbpath', '/content/mdb', '--port', '27017', '--bind_ip', '127.0.0.1'],
                     stdout=open('/content/mongod.log','w'), stderr=subprocess.STDOUT)
    from pymongo import MongoClient
    c = MongoClient('mongodb://localhost:27017', serverSelectionTimeoutMS=8000)
    for _ in range(20):
        try: c.admin.command('ping'); return c
        except Exception: time.sleep(1)
    raise RuntimeError('mongod did not become ready')

client, BACKEND = None, None
try:
    client = start_real_mongo(); BACKEND = 'real MongoDB 7.0 (mongod @ localhost:27017)'
except Exception as e:
    print('Real mongod unavailable -> using mongomock. Reason:', repr(e)[:140])
    import mongomock; client = mongomock.MongoClient(); BACKEND = 'mongomock (in-memory, pymongo-compatible)'
print('Backend:', BACKEND)

### Build one document per hour (nested measures, fuels, flags)

In [ ]:
df = pd.read_parquet(STAGING_PARQUET)
df['utc_timestamp'] = pd.to_datetime(df['utc_timestamp'], utc=True)
df = df.sort_values('utc_timestamp').reset_index(drop=True)

fuel_cols = [c for c in df.columns if c.startswith('net_generation_mw_from_')]
has_imp   = 'is_imputed' in df.columns
has_night = 'solar_night_flag' in df.columns
print(f'rows {len(df):,} | fuel cols {len(fuel_cols)} | is_imputed={has_imp} | solar_night_flag={has_night}')

def fnum(v):
    return None if (v is None or (isinstance(v, float) and np.isnan(v))) else float(v)

def build_doc(r):
    ts = r['utc_timestamp']
    doc = {
        '_id': ts.isoformat(),
        'ts':  ts.isoformat(),
        'ba':  'ERCO',
        'demand_mw':            fnum(r.get('demand_mw')),
        'forecast_mw':          fnum(r.get('demand_forecast_mw')),
        'net_generation_mw':    fnum(r.get('net_generation_mw')),
        'total_interchange_mw': fnum(r.get('total_interchange_mw')),
        'time': {'year': int(ts.year), 'month': int(ts.month), 'day': int(ts.day),
                 'hour': int(ts.hour), 'dow': int(ts.weekday())},
        'flags': {
            'imputed':     bool(r['is_imputed'])       if has_imp   and pd.notna(r.get('is_imputed'))       else False,
            'solar_night': bool(r['solar_night_flag']) if has_night and pd.notna(r.get('solar_night_flag')) else False,
        },
    }
    fuels = {}
    for c in fuel_cols:
        v = r.get(c)
        if pd.notna(v): fuels[c.replace('net_generation_mw_from_', '')] = float(v)
    doc['fuels'] = fuels                 # empty {} for pre-2024 rows (fuel columns null by design)
    return doc

docs = [build_doc(r) for r in df.to_dict('records')]
print('built', len(docs), 'documents')

### Load into the collection + indexes

In [ ]:
db = client[DB_NAME]; col = db[COLL_NAME]
col.drop()
col.insert_many(docs, ordered=False)
col.create_index('ts', unique=True)
col.create_index([('time.year', 1), ('time.month', 1)])
col.create_index('flags.imputed')
print('inserted:', col.count_documents({}), 'documents into', f'{DB_NAME}.{COLL_NAME}')

### Validate — counts, sample document, indexes

In [ ]:
n       = col.count_documents({})
n_imp   = col.count_documents({'flags.imputed': True})
n_night = col.count_documents({'flags.solar_night': True})
sample  = col.find_one({'time.year': {'$gte': 2024}, 'fuels.natural_gas': {'$exists': True}}) or col.find_one()
print(f'documents={n:,}  imputed={n_imp}  solar_night={n_night}')
print('indexes:', list(col.index_information().keys()))
print('\nsample document:')
print(json.dumps(sample, indent=2, default=str))

### Aggregation-pipeline queries

In [ ]:
by_year = list(col.aggregate([
    {'$group': {'_id': '$time.year', 'avg_demand': {'$avg': '$demand_mw'}, 'hours': {'$sum': 1}}},
    {'$sort': {'_id': 1}}]))
peak = list(col.aggregate([
    {'$match': {'demand_mw': {'$ne': None}}}, {'$sort': {'demand_mw': -1}}, {'$limit': 1}]))[0]
fuelmix = list(col.aggregate([
    {'$match': {'time.year': {'$gte': 2024}}},
    {'$group': {'_id': None, 'gas': {'$avg': '$fuels.natural_gas'}, 'wind': {'$avg': '$fuels.wind'},
                'solar': {'$avg': '$fuels.solar'}, 'nuclear': {'$avg': '$fuels.nuclear'}}}]))

print('Avg demand by year:')
for r in by_year: print(f"  {r['_id']}: {r['avg_demand']:.0f} MW  ({r['hours']} h)")
print(f"\nPeak demand: {peak['demand_mw']:.0f} MW at {peak['ts']}")
if fuelmix:
    print('Avg generation 2024+ (MW):', {k: round(v) for k, v in fuelmix[0].items() if k != '_id' and v is not None})


### Write validation summary (parity with the Neo4J output)

In [ ]:
L = []
L.append('=== MongoDB Validation ===')
L.append(f'Backend     : {BACKEND}')
L.append(f'Database    : {DB_NAME}')
L.append(f'Collection  : {COLL_NAME}')
L.append(f'Documents   : {n:,}')
L.append(f'Imputed     : {n_imp}')
L.append(f'Solar-night : {n_night}')
L.append('Indexes     : _id, ts (unique), (time.year, time.month), flags.imputed')
L.append('')
L.append('Avg demand by year:')
for r in by_year: L.append(f"  {r['_id']} : {r['avg_demand']:.0f} MW  ({r['hours']} h)")
L.append(f"Peak demand : {peak['demand_mw']:.0f} MW at {peak['ts']}")
L.append('')
L.append('Sample document:')
L.append(json.dumps(sample, indent=2, default=str))
out = '\n'.join(L)
(BI_DIR / 'mongodb_validation.txt').write_text(out)
print(out)
print('\nSaved -> BI/mongodb_validation.txt')

### Done
MongoDB document store built and validated on the full 5-year series — parity with the Neo4J graph. Copy the printed block into a **“mongodb output”** Google Doc in the `noSQL` folder so both NoSQL stores are documented side by side.